# 🚀 SageMaker Keras Training Orchestration

While the `deploy/` directory defines the cloud execution environment (Docker and scripts), the `notebooks/` directory serves as the orchestration headquarters to manage experiments from a local machine or a SageMaker Studio instance.

In this notebook, we will:
1. Launch training jobs remotely using the `sagemaker` SDK.
2. Deploy the custom `Dockerfile` and training code to an AWS GPU cluster.
3. Securely inject credentials (like ClearML API keys) without hardcoding them in our scripts.

In [ ]:
%pip install sagemaker boto3 python-dotenv -q

import os
import sagemaker
from sagemaker.estimator import Estimator
from dotenv import load_dotenv

# Load secrets from a local .env file.
# Ensure this file is never committed to Git.
load_dotenv('../.env')

print(f"SageMaker Version: {sagemaker.__version__}")

### 1. Configure the Role and Environment Vars

By using the `.fit()` method of the `sagemaker` Estimator, we can delegate the heavy GPU model training to SageMaker without requiring a powerful local machine.

During this process, ClearML credentials (API/Access/Secret Keys) are loaded from the **local environment (this notebook)** via `.env`. They are securely injected into the cloud Estimator without leaving any traces in the hardcoded Docker image.

In [ ]:
# 1. Load AWS Credentials / Role 
# (On SageMaker Studio, get_execution_role() works automatically)
role = "arn:aws:iam::123456789012:role/service-role/AmazonSageMaker-ExecutionRole-20230101T000000"

# 2. Define Image URI (from deploying your Dockerfile beforehand, typically in ECR)
image_uri = "123456789012.dkr.ecr.us-east-1.amazonaws.com/crowdnav-training-env:latest"

# 3. Create SageMaker Estimator pointing to custom Keras Container
yolo_estimator = Estimator(
    image_uri=image_uri,
    role=role,
    instance_count=1,
    instance_type="ml.g4dn.xlarge", # Cost-effective entry GPU instance
    output_path="s3://my-crowdnav-bucket/models/",
    # ==== SECURE CLEARML INJECTION ====
    environment={
        "CLEARML_WEB_HOST": "https://app.clear.ml",
        "CLEARML_API_HOST": "https://api.clear.ml",
        "CLEARML_FILES_HOST": "https://files.clear.ml",
        "CLEARML_API_ACCESS_KEY": os.environ.get("CLEARML_API_ACCESS_KEY"),
        "CLEARML_API_SECRET_KEY": os.environ.get("CLEARML_API_SECRET_KEY")
    }
)

### 2. Trigger the Training Run

Specify the data path in S3 and launch the model onto the AWS infrastructure using the pre-configured environment. Once this job starts, training continues safely in the cloud—even if your local notebook session stops or logs out—and continuously streams logs to the ClearML dashboard.

In [ ]:
# 4. Trigger Training directly to AWS SageMaker
# The S3 data below will be downloaded and extracted to /opt/ml/input/data/training inside the container.
yolo_estimator.fit(
    {"training": "s3://my-crowdnav-bucket/dataset/yolo_format/"}
)